In [73]:
#Creating the Multithreaded Cracker Source Code
%%bash
cat > CrackAZ99_threads.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <crypt.h>
#include <pthread.h>
#include <time.h>

typedef struct {
  const char *salt;
  const char *target;
  long long startIndex;
  long long endIndex;
  int threadId;
} CrackTask;

static volatile int passwordFound = 0;
static char foundPlain[8];
static pthread_mutex_t foundMutex = PTHREAD_MUTEX_INITIALIZER;

static void indexToPlain(long long idx, char out[7]) {
  int number = (int)(idx % 100);
  int letterIndex = (int)(idx / 100);
  int x = letterIndex / 26;
  int y = letterIndex % 26;
  sprintf(out, "%c%c%02d", 'A' + x, 'A' + y, number);
}

// extract salt up to the 3rd '$' (e.g., "$6$AS$")
static void getSalt(char *saltOut, const char *hash) {
  int dollars = 0, i = 0;
  while (hash[i] && dollars < 3) {
    saltOut[i] = hash[i];
    if (hash[i] == '$') dollars++;
    i++;
  }
  saltOut[i] = '\0';
}

static void *crackSlice(void *arg) {
  CrackTask *task = (CrackTask *)arg;
  char candidate[7];
  struct crypt_data data;
  data.initialized = 0;

  for (long long i = task->startIndex; i <= task->endIndex; i++) {
    if (passwordFound) break;

    indexToPlain(i, candidate);

    char *enc = crypt_r(candidate, task->salt, &data);
    if (enc && strcmp(enc, task->target) == 0) {
      pthread_mutex_lock(&foundMutex);
      if (!passwordFound) {
        passwordFound = 1;
        strcpy(foundPlain, candidate);
      }
      pthread_mutex_unlock(&foundMutex);
      break;
    }
  }
  return NULL;
}

static void buildSlices(long long totalCount, int threadCount,
                        long long *starts, long long *ends) {
  long long base = totalCount / threadCount;
  long long rem = totalCount % threadCount;
  long long nextStart = 0;
  for (int t = 0; t < threadCount; t++) {
    long long size = base + (t < rem ? 1 : 0);
    starts[t] = nextStart;
    ends[t] = nextStart + size - 1;
    nextStart = ends[t] + 1;
  }
}

int main(int argc, char **argv) {
  if (argc < 3) {
    fprintf(stderr, "Usage: %s SALT_AND_HASH THREAD_COUNT\n", argv[0]);
    return 1;
  }
  const char *saltAndHash = argv[1];
  int threadCount = atoi(argv[2]);
  if (threadCount <= 0) {
    fprintf(stderr, "THREAD_COUNT must be > 0\n");
    return 1;
  }

  char salt[64];
  getSalt(salt, saltAndHash);

  const long long total = 26LL * 26LL * 100LL; // 67600
  long long *starts = (long long*)malloc(sizeof(long long)*threadCount);
  long long *ends   = (long long*)malloc(sizeof(long long)*threadCount);
  pthread_t *threads = (pthread_t*)malloc(sizeof(pthread_t)*threadCount);
  CrackTask *tasks   = (CrackTask*)malloc(sizeof(CrackTask)*threadCount);
  if (!starts || !ends || !threads || !tasks) {
    fprintf(stderr, "Allocation failed\n");
    return 1;
  }

  buildSlices(total, threadCount, starts, ends);

  struct timespec t0, t1;
  clock_gettime(CLOCK_MONOTONIC, &t0);

  for (int t = 0; t < threadCount; t++) {
    tasks[t].salt = salt;
    tasks[t].target = saltAndHash;
    tasks[t].startIndex = starts[t];
    tasks[t].endIndex = ends[t];
    tasks[t].threadId = t;
    pthread_create(&threads[t], NULL, crackSlice, &tasks[t]);
  }

  for (int t = 0; t < threadCount; t++) {
    pthread_join(threads[t], NULL);
  }

  clock_gettime(CLOCK_MONOTONIC, &t1);
  double elapsed = (t1.tv_sec - t0.tv_sec) + (t1.tv_nsec - t0.tv_nsec) / 1e9;

  if (passwordFound) {
    printf("FOUND: %s\n", foundPlain);
  } else {
    printf("NOT FOUND\n");
  }
  printf("Time: %.6f s with %d threads\n", elapsed, threadCount);

  free(starts); free(ends); free(threads); free(tasks);
  return 0;
}
EOF

In [74]:
%%bash
cat > EncryptSHA512.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <crypt.h>

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Usage: %s PASSWORD\n", argv[0]);
        return 1;
    }
    const char *password = argv[1];
    // Using a fixed salt for consistent hashing for testing
    const char *salt = "$6$AS$"; // SHA-512 with a fixed salt

    char *hashed = crypt(password, salt);
    if (hashed == NULL) {
        perror("crypt");
        return 1;
    }
    printf("%s\n", hashed);
    return 0;
}
EOF

cat > Slicing.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char **argv) {
    if (argc < 3) {
        fprintf(stderr, "Usage: %s TOTAL_COUNT THREAD_COUNT\n", argv[0]);
        return 1;
    }
    long long totalCount = atoll(argv[1]);
    int threadCount = atoi(argv[2]);

    if (totalCount <= 0 || threadCount <= 0) {
        fprintf(stderr, "TOTAL_COUNT and THREAD_COUNT must be > 0\n");
        return 1;
    }

    long long base = totalCount / threadCount;
    long long rem = totalCount % threadCount;
    long long nextStart = 0;

    for (int t = 0; t < threadCount; t++) {
        long long size = base + (t < rem ? 1 : 0);
        long long start = nextStart;
        long long end = nextStart + size - 1;
        printf("thread %d: start = %lld  end = %lld\n", t, start, end);
        nextStart = end + 1;
    }

    return 0;
}
EOF

In [75]:
# Compile all three programs:
# - encrypt: Password encryption utility using SHA-512
# - crack: Multithreaded password cracker with dynamic slicing
# - slicing: Utility to demonstrate work distribution across threads
%%bash
gcc -O3 EncryptSHA512.c -lcrypt -o encrypt
gcc -O3 CrackAZ99_threads.c -pthread -lcrypt -o crack
gcc -O3 Slicing.c -o slicing

In [76]:
#Generate Test Hash
%%bash
./encrypt HP93

$6$AS$Ig.vW9RG9J5gPFUvHwyV67GdVVndF.2ROH6.qZjQN1Nm5kqn0t/FKNf4.48qRHdyAWwIQOtKkCosTrwyj3SvJ.


In [77]:
# Use the complete hash from cell 4
%%bash
HASH='$6$AS$Ig.vW9RG9J5gPFUvHwyV67GdVVndF.2ROH6.qZjQN1Nm5kqn0t/FKNf4.48qRHdyAWwIQOtKkCosTrwyj3SvJ.'
NEWHASH="$(./encrypt HP93)"
echo "Original: $HASH"
echo "New:      $NEWHASH"
test "$HASH" = "$NEWHASH" && echo "Hashes match" || echo "Hashes differ"

Original: $6$AS$Ig.vW9RG9J5gPFUvHwyV67GdVVndF.2ROH6.qZjQN1Nm5kqn0t/FKNf4.48qRHdyAWwIQOtKkCosTrwyj3SvJ.
New:      $6$AS$Ig.vW9RG9J5gPFUvHwyV67GdVVndF.2ROH6.qZjQN1Nm5kqn0t/FKNf4.48qRHdyAWwIQOtKkCosTrwyj3SvJ.
Hashes match


In [78]:
%%bash
# Add debug output to see what salt is being extracted
echo "Hash: $HASH"
echo "Expected salt: \$6\$AS\$"

Hash: 
Expected salt: $6$AS$


In [79]:
%%bash
# Re-generate the HP93 hash properly
HASH="$(./encrypt HP93)"
echo "Generated hash: $HASH"
echo "Hash length: ${#HASH}"

Generated hash: $6$AS$Ig.vW9RG9J5gPFUvHwyV67GdVVndF.2ROH6.qZjQN1Nm5kqn0t/FKNf4.48qRHdyAWwIQOtKkCosTrwyj3SvJ.
Hash length: 92


In [80]:
%%bash
# Test with AA00 to make sure basic functionality works
AA00HASH="$(./encrypt AA00)"
./crack "$AA00HASH" 8

FOUND: AA00
Time: 0.035359 s with 8 threads


In [81]:
%%bash
# Re-generate HP93 hash and store it correctly
HASH="$(./encrypt HP93)"
echo "Generated hash: $HASH"
echo "Hash length: ${#HASH}"


Generated hash: $6$AS$Ig.vW9RG9J5gPFUvHwyV67GdVVndF.2ROH6.qZjQN1Nm5kqn0t/FKNf4.48qRHdyAWwIQOtKkCosTrwyj3SvJ.
Hash length: 92


In [82]:
%%bash
# Generate HP93 hash fresh and then test the cracker
HASH="$(./encrypt HP93)"
echo "Generated hash for cracking: $HASH"
./crack "$HASH" 8

Generated hash for cracking: $6$AS$Ig.vW9RG9J5gPFUvHwyV67GdVVndF.2ROH6.qZjQN1Nm5kqn0t/FKNf4.48qRHdyAWwIQOtKkCosTrwyj3SvJ.
FOUND: HP93
Time: 72.256306 s with 8 threads


In [83]:
# Calculate the expected index for "HP93" and print what indexToPlain would produce for it.
%%bash
echo "Expected index for HP93: 19793"
# Compile a small C program to test indexToPlain
cat > test_indexToPlain.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>

static void indexToPlain(long long idx, char out[7]) {
  int number = (int)(idx % 100);
  int letterIndex = (int)(idx / 100);
  int x = letterIndex / 26;
  int y = letterIndex % 26;
  sprintf(out, "%c%c%02d", 'A' + x, 'A' + y, number);
}

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Usage: %s INDEX\n", argv[0]);
        return 1;
    }
    long long index = atoll(argv[1]);
    char plain[7];
    indexToPlain(index, plain);
    printf("indexToPlain(%lld) -> %s\n", index, plain);
    return 0;
}
EOF

gcc test_indexToPlain.c -o test_indexToPlain
./test_indexToPlain 19793

Expected index for HP93: 19793
indexToPlain(19793) -> HP93


In [84]:
#Demonstrate Dynamic Slicing
%%bash
./slicing 67600 8

thread 0: start = 0  end = 8449
thread 1: start = 8450  end = 16899
thread 2: start = 16900  end = 25349
thread 3: start = 25350  end = 33799
thread 4: start = 33800  end = 42249
thread 5: start = 42250  end = 50699
thread 6: start = 50700  end = 59149
thread 7: start = 59150  end = 67599


In [85]:
#Test with Known Password (AA00)
%%bash
gcc -O3 EncryptSHA512.c -lcrypt -o encrypt
TESTHASH="$(./encrypt AA00)"
./crack "$TESTHASH" 8

FOUND: AA00
Time: 0.041938 s with 8 threads
